# Bigdata Research Tools - Tutorial

This notebook demonstrates all the key functionality of the `bigdata-research-tools` library with practical, working examples.

## Table of Contents

1. Setup and Authentication
2. Basic Search Examples
   - Search by Companies
   - Custom Query Search
3. Next Steps

---


## 1. Setup and Authentication

First, let's import the necessary libraries and set up authentication.

**⚠️ Important**: Run all cells in order, starting from the top. Each cell depends on the previous ones.


In [4]:
# Import core libraries
import logging
import pandas as pd
import numpy as np
from typing import List, Dict
from dotenv import load_dotenv

# Bigdata client imports
from bigdata_client.models.search import DocumentType, SortBy
from bigdata_client.daterange import AbsoluteDateRange

# Bigdata research tools imports
from bigdata_research_tools.client import bigdata_connection
from bigdata_research_tools.search.screener_search import search_by_companies
from bigdata_research_tools.search.search import run_search
from bigdata_research_tools.search.query_builder import (
    build_batched_query,
    EntitiesToSearch,
    create_date_ranges
)
from bigdata_research_tools.workflows import NarrativeMiner, ThematicScreener
from bigdata_research_tools.workflows.risk_analyzer import RiskAnalyzer
from bigdata_research_tools.portfolio.portfolio_constructor import (
    PortfolioConstructor, WeightMethod
)

# Configure clean logging for notebook
logging.basicConfig(
    level=logging.INFO,
    format='%(levelname)s: %(message)s'
)
logger = logging.getLogger(__name__)

print("✅ Libraries imported successfully")


✅ Libraries imported successfully


In [5]:
# Load environment variables and connect to Bigdata API
load_dotenv()
print(f"Environment variables loaded: {load_dotenv()}")

# Establish connection to Bigdata API
bigdata = bigdata_connection()
print("✅ Connected to Bigdata API successfully")

Environment variables loaded: True
✅ Connected to Bigdata API successfully


---

## Basic Search Examples

Let's start with the fundamental search capabilities.

### Search by Companies

This example shows how to search for documents mentioning specific companies and topics.


In [6]:
def demo_search_by_companies():
    """Demonstrate search_by_companies functionality."""
    
    print("🔍 Search by Companies Example")
    print("=" * 40)
    
    # Get companies using ticker symbols (reliable method)
    tickers = ["AAPL", "MSFT", "TSLA"]
    companies = []
    
    print("Finding companies using tickers...")
    for ticker in tickers:
        results = bigdata.knowledge_graph.autosuggest(ticker, limit=1)
        if results:
            companies.extend(results)
            print(f"  ✅ Found: {results[0].name} ({ticker})")
    
    if not companies:
        print("❌ No companies found")
        return None
    
    # Define search sentences
    sentences = ["AI is transforming business", "Cloud adoption is accelerating"]
    
    print(f"\n📊 Searching for content across {len(companies)} companies...")
    
    try:
        # Search for documents
        results_df = search_by_companies(
            companies=companies,
            sentences=sentences,
            start_date="2024-01-01",
            end_date="2024-06-30",
            scope=DocumentType.NEWS,
            document_limit=20,
            batch_size=5
        )
        
        # Display results
        if not results_df.empty:
            print(f"\n✅ Found {len(results_df)} relevant documents")
            
            # Company breakdown
            company_counts = results_df['entity_name'].value_counts()
            print("\n📈 Documents by company:")
            for company, count in company_counts.items():
                print(f"  {company}: {count} documents")
            
            # Sample headlines
            print("\n📰 Sample headlines:")
            for headline in results_df['headline'].head(3):
                print(f"  • {headline}")
            
            return results_df
        else:
            print("⚠️ No documents found")
            return None
            
    except ValueError as e:
        if "No rows to process" in str(e):
            print("⚠️ No documents found matching the search criteria")
            print("💡 Try different date ranges or search terms")
            return None
        else:
            raise

# Run the example
search_results = demo_search_by_companies()


🔍 Search by Companies Example
Finding companies using tickers...
  ✅ Found: Apple Inc. (AAPL)
  ✅ Found: Microsoft Corp. (MSFT)
  ✅ Found: Tesla Inc. (TSLA)

📊 Searching for content across 3 companies...


INFO: About to run 12 queries
Processing news results...: 100%|██████████████████████████████████████████████████████████████████████████| 240/240 [00:00<00:00, 14639.59it/s]



✅ Found 306 relevant documents

📈 Documents by company:
  Microsoft Corp.: 294 documents
  Apple Inc.: 12 documents

📰 Sample headlines:
  • Alphabet (GOOGL) Set to Invest $1 Billion in U.K. Data Center
  • Cloud Infrastructure Service Market Predicted to Garner USD 282.1 Bn By 2033, At CAGR 8.11% | Report by Marketresearch.biz
  • Are These AI Stocks Ready to Rip or Take a Back Seat?


In [7]:
# Display the results DataFrame if we got results
if search_results is not None:
    print("📊 Results DataFrame:")
    display(search_results.head(10))
else:
    print("No results to display")


📊 Results DataFrame:


,timestamp_utc,document_id,sentence_id,headline,entity_id,document_type,is_reporting_entity,entity_name,entity_sector,entity_industry,entity_country,entity_ticker,text,other_entities,entities,masked_text,other_entities_map
0,2024-01-20 00:00:00+00:00,3B785A6E8BD64B5D2CB1F2CF74299AF6,3B785A6E8BD64B5D2CB1F2CF74299AF6-14,Alphabet (GOOGL) Set to Invest $1 Billion in U...,228D42,news,False,Microsoft Corp.,Technology,Software,United States,MSFT,"Meanwhile, Microsoft Azure became Microsoft's ...",,"[{'key': '228D42', 'name': 'Microsoft Corp.', ...","Meanwhile, Target Company Azure became Target ...",None
1,2024-01-23 07:34:27+00:00,1F3B5215FC2C43FBD255CB364508356D,1F3B5215FC2C43FBD255CB364508356D-13,Cloud Infrastructure Service Market Predicted ...,228D42,news,False,Microsoft Corp.,Technology,Software,United States,MSFT,North America commands a significant 42.30% sh...,"Alphabet Inc., Deutsche Telekom AG, Amazon Web...","[{'key': '228D42', 'name': 'Microsoft Corp.', ...",North America commands a significant 42.30% sh...,"[(1, Deutsche Telekom AG), (2, Alphabet Inc.),..."
2,2024-01-24 00:00:00+00:00,09FB2895B76C059AA0E7488071AAE2FE,09FB2895B76C059AA0E7488071AAE2FE-3,Are These AI Stocks Ready to Rip or Take a Bac...,228D42,news,False,Microsoft Corp.,Technology,Software,United States,MSFT,"""AI's sweeping applicability has the potential...",NVIDIA Corp.,"[{'key': '228D42', 'name': 'Microsoft Corp.', ...","""AI's sweeping applicability has the potential...","[(4, NVIDIA Corp.)]"
3,2024-01-25 08:02:14+00:00,B48F167CFDD14FEFF7514A59FD3A0729,B48F167CFDD14FEFF7514A59FD3A0729-7,Data Lake Market to Hit US$ 41.2 Billion by 20...,228D42,news,False,Microsoft Corp.,Technology,Software,United States,MSFT,Key Companies Profiled Microsoft Corporation D...,,"[{'key': '228D42', 'name': 'Microsoft Corp.', ...",Key Companies Profiled Target Company Data Lak...,None
4,2024-01-25 16:48:43+00:00,2AA84830085B76F18E3B995082D07C47,2AA84830085B76F18E3B995082D07C47-11,Eviden and Microsoft forge five-year global st...,228D42,news,False,Microsoft Corp.,Technology,Software,United States,MSFT,"Judson Althoff, Executive Vice President and C...","Eviden SAS, OpenAI Inc.","[{'key': 'HWHMXA', 'name': 'Eviden SAS', 'tick...","Judson Althoff, Executive Vice President and C...","[(5, OpenAI Inc.), (6, Eviden SAS)]"
5,2024-01-25 16:48:43+00:00,2AA84830085B76F18E3B995082D07C47,2AA84830085B76F18E3B995082D07C47-2,Eviden and Microsoft forge five-year global st...,228D42,news,False,Microsoft Corp.,Technology,Software,United States,MSFT,"PARIS, Jan. 25, 2024 /PRNewswire/ -- Eviden, t...","Eviden SAS, Eviden SAS, Atos S.E.","[{'key': '228D42', 'name': 'Microsoft Corp.', ...","PARIS, Jan. 25, 2024 /PRNewswire/ -- Other Com...","[(6, Eviden SAS), (7, Atos S.E.), (6, Eviden S..."
6,2024-01-25 16:48:43+00:00,2AA84830085B76F18E3B995082D07C47,2AA84830085B76F18E3B995082D07C47-4,Eviden and Microsoft forge five-year global st...,228D42,news,False,Microsoft Corp.,Technology,Software,United States,MSFT,Both organizations are committed to driving an...,"Eviden SAS, Eviden SAS, Eviden SAS","[{'key': 'HWHMXA', 'name': 'Eviden SAS', 'tick...",Both organizations are committed to driving an...,"[(6, Eviden SAS), (6, Eviden SAS), (6, Eviden ..."
7,2024-01-25 16:48:43+00:00,2AA84830085B76F18E3B995082D07C47,2AA84830085B76F18E3B995082D07C47-8,Eviden and Microsoft forge five-year global st...,228D42,news,False,Microsoft Corp.,Technology,Software,United States,MSFT,Accelerating SAP Transformation: Built on 14+ ...,"Eviden SAS, Eviden SAS, SAP SE, SAP SE, SAP SE...","[{'key': 'HWHMXA', 'name': 'Eviden SAS', 'tick...",Accelerating Other Company_8 Transformation: B...,"[(8, SAP SE), (8, SAP SE), (8, SAP SE), (6, Ev..."
8,2024-01-25 16:48:43+00:00,2AA84830085B76F18E3B995082D07C47,2AA84830085B76F18E3B995082D07C47-1,Eviden and Microsoft forge five-year global st...,228D42,news,False,Microsoft Corp.,Technology,Software,United States,MSFT,Five-year agreement expected to drive $2.8B US...,"Eviden SAS, Eviden SAS

### 2. Custom Query Search

This example demonstrates building custom queries and using the run_search function.


In [10]:
def demo_run_search():
    """Demonstrate run_search with custom query building."""
    
    print("🔧 Custom Query Search Example")
    print("=" * 40)
    
    # Define entities to search for
    entities = EntitiesToSearch(
        companies=["Apple Inc", "Google", "Microsoft Corp"],
        topic=["earnings", "financial results"],
        concepts=["revenue growth", "profit margins"]
    )
    
    # Define search sentences
    sentences = [
        "quarterly earnings performance",
        "revenue growth and profitability"
    ]
    
    print("🔨 Building search queries...")
    
    # Build queries
    queries = build_batched_query(
        sentences=sentences,
        keywords=["earnings", "revenue", "profit"],
        entities=entities,
        control_entities=None,
        sources=None,
        batch_size=5,
        fiscal_year=None,
        scope=DocumentType.NEWS,
        custom_batches=None
    )
    
    print(f"✅ Generated {len(queries)} search queries")
    
    # Create date ranges
    date_ranges = create_date_ranges("2024-10-01", "2024-12-31", "M")
    print(f"📅 Searching across {len(date_ranges)} time periods")
    
    # Execute search
    print("🔍 Executing search...")
    
    search_results = run_search(
        queries=queries,
        date_ranges=date_ranges,
        scope=DocumentType.NEWS,
        limit=8,
        only_results=True
    )
    
    # Process results
    all_documents = []
    
    for result_batch in search_results:
        for doc in result_batch:
            # Convert timezone-aware datetime for compatibility
            timestamp_naive = doc.timestamp.replace(tzinfo=None) if doc.timestamp else None
            
            doc_data = {
                'timestamp': timestamp_naive,
                'headline': doc.headline,
                'source': doc.source.name if doc.source else 'Unknown',
                'doc_id': doc.id if hasattr(doc, 'id') else 'N/A'
            }
            all_documents.append(doc_data)
    
    # Convert to DataFrame
    results_df = pd.DataFrame(all_documents)
    
    if not results_df.empty:
        print(f"\n✅ Found {len(results_df)} documents total")
        
        # Source distribution
        source_counts = results_df['source'].value_counts()
        print("\n📊 Documents by source:")
        for source, count in source_counts.head(5).items():
            print(f"  {source}: {count} documents")
        
        # Sample headlines
        print("\n📰 Sample headlines:")
        for headline in results_df['headline'].head(3):
            print(f"  • {headline}")
        
        return results_df
    else:
        print("⚠️ No documents found")
        return None

# Run the example
custom_search_results = demo_run_search()

🔧 Custom Query Search Example
🔨 Building search queries...
✅ Generated 2 search queries
📅 Searching across 3 time periods
🔍 Executing search...


Querying Bigdata...: 100%|████████████████████████████████████████████████████████████████████████████████████████| 6/6 [00:03<00:00,  1.73it/s]



✅ Found 48 documents total

📊 Documents by source:
  Benzinga: 23 documents
  Nasdaq: 7 documents
  Associated Press: 4 documents
  AOL.com: 3 documents
  Yahoo! Finance: 3 documents

📰 Sample headlines:
  • Intel Stock Climbs On Better-Than-Expected Q3 Results: Details
  • Amcor reports first quarter result and reaffirms outlook for fiscal 2025
  • Zacks Earnings Trends Highlights: Tesla, Alphabet and Microsoft


In [11]:
# Display the custom_search_results DataFrame if we got results
if custom_search_results is not None:
    print("📊 Results DataFrame:")
    display(custom_search_results.head(10))
else:
    print("No results to display")

📊 Results DataFrame:


,timestamp,headline,source,doc_id
0,2024-10-31 20:37:50,Intel Stock Climbs On Better-Than-Expected Q3 ...,Benzinga,94AD5D1B68EEBB1F14CED1D63C521202
1,2024-10-31 20:10:08,Amcor reports first quarter result and reaffir...,Benzinga,CD9C477A343492A6A9B422D97DF79B3F
2,2024-10-31 15:59:02,"Zacks Earnings Trends Highlights: Tesla, Alpha...",Nasdaq,3033ABB7F82D6E4B615A8DE37A359A9C
3,2024-10-31 20:07:25,Dorman Products raises FY24 adjusted EPS view ...,The Fly,B7169021B33F8633BA1EC5666EF888A2
4,2024-10-31 12:09:56,Charles River Associates (CRA) Reports Financi...,Associated Press,F6201B5961814B08C970C0CE493E04E3
5,2024-10-31 12:25:12,How To Earn $500 A Month From Apple Stock Ahea...,Benzinga,49970CDC82D324F91ED96D7ACC0AEE32
6,2024-10-31 20:10:37,Onto Innovation Reports 2024 Third Quarter Res...,Associated Press,EBF7CC0983298CFEE339529E9C7815A9
7,2024-10-31 22:53:50,Apple (AAPL) Beats Q4 Earnings and Revenue Est...,Nasdaq,DA8407CCDA767FB391E50DCF71F99645
8,2024-12-31 18:59:53,Walgreens Earnings Are Imminent; These Most Ac...,Benzinga,A54326C95D1C86D5E7F4D6CF7ACDE668
9,2024-12-31 13:45:36,"Nvidia, Goldman Sachs, Jefferies Financial And...",Benzinga,0F5C17E1B745BC694E9CC00EE7859142


## 3. Next Steps

This notebook has demonstrated the key capabilities of the Bigdata Research Tools library:

✅ **Basic Search Functions**: `search_by_companies()` and `run_search()`  

### Explore

1. **Customize Examples**: Modify the company lists, date ranges, and search terms for your specific use case
2. **Combine Workflows**: Use multiple tools together for comprehensive analysis
3. **Scale Up**: Increase batch sizes and document limits for production use
4. **Export Results**: Save DataFrames to Excel/CSV for further analysis

### Additional Workflows

 The library also offers a suite of advanced, AI-powered workflows to supercharge your research:
 
 - **NarrativeMiner**: Uncover and track the evolution of narratives over time.
 - **ThematicScreener**: Analyze how companies are exposed to specific themes.
 - **RiskAnalyzer**: Assess risk exposure using customizable taxonomies.
 
 For even more practical examples, explore our [Bigdata Cookbook](https://github.com/Bigdata-com/bigdata-cookbook), which features a collection of ready-to-use notebooks for a variety of finance-related guided workflows.

*Note: These require LLM access (OpenAI API key) and may take longer to run.*

For more detailed documentation, see the [USER_GUIDE.md](README.md) file.
